# Which team stats best predict NBA wins?

**Step 3: train two models with a time-based split.**

Each row is one game from the home team's side. Features are home-minus-away differences (built in `src/build_features.py`). Target: did the home team win?

In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score, log_loss

df = pd.read_csv('../data/processed/game_features.csv')
FEATURES = ['EFG_PCT_DIFF', 'TOV_PCT_DIFF', 'OREB_PCT_DIFF', 'FT_RATE_DIFF',
            'REB_MARGIN', 'FG3A_RATE_DIFF', 'AST_RATE_DIFF',
            'STL_PER100_DIFF', 'BLK_PER100_DIFF', 'PACE', 'HOME']
df.groupby('SEASON').size().rename('games').to_frame().T

SEASON,2016-17,2017-18,2018-19,2019-20,2020-21,2021-22,2022-23,2023-24,2024-25,2025-26
games,1230,1230,1230,1059,1080,1230,1230,1230,1230,1230


## Time-based split (no random shuffling)

Train on the 7 earlier seasons and test on the 3 most recent ones. A random split would mix future games into training, so the model could pick up league-wide trends (like the rise of the 3-pointer) it wouldn't actually know in advance. Splitting by time copies how the model would be used for real.

In [2]:
TEST_SEASONS = ['2023-24', '2024-25', '2025-26']
train = df[~df.SEASON.isin(TEST_SEASONS)]
test = df[df.SEASON.isin(TEST_SEASONS)]
X_train, y_train = train[FEATURES], train['WIN']
X_test, y_test = test[FEATURES], test['WIN']
print(f'Train: {len(train):,} games ({train.SEASON.min()} to {train.SEASON.max()})')
print(f'Test:  {len(test):,} games ({test.SEASON.min()} to {test.SEASON.max()})')

Train: 8,289 games (2016-17 to 2022-23)
Test:  3,690 games (2023-24 to 2025-26)


## Models

- **Logistic regression**: simple and easy to explain. Features are standardized first so their coefficients can be compared directly.
- **Random forest**: many decision trees voting together. It can pick up non-linear effects and interactions.
- **Baseline**: always pick the home team. Any model has to beat this to be worth anything.

In [3]:
logreg = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
forest = RandomForestClassifier(n_estimators=400, min_samples_leaf=20,
                                random_state=42, n_jobs=-1)
logreg.fit(X_train, y_train)
forest.fit(X_train, y_train)

rows = []
base_p = np.full(len(y_test), y_train.mean())
rows.append(('Baseline: always pick home team', accuracy_score(y_test, np.ones(len(y_test))),
             0.5, log_loss(y_test, base_p)))
for name, model in [('Logistic regression', logreg), ('Random forest', forest)]:
    p = model.predict_proba(X_test)[:, 1]
    rows.append((name, accuracy_score(y_test, p > 0.5), roc_auc_score(y_test, p), log_loss(y_test, p)))
results = pd.DataFrame(rows, columns=['Model', 'Accuracy', 'ROC AUC', 'Log loss']).set_index('Model')
results.round(3)

,Accuracy,ROC AUC,Log loss
Model,,,
Baseline: always pick home team,0.547,0.500,0.690
Logistic regression,0.960,0.994,0.096
Random forest,0.930,0.985,0.217


**How to read this:** *Accuracy* is the share of test games called correctly. *ROC AUC* measures how well the model ranks winners above losers (0.5 = coin flip, 1.0 = perfect). *Log loss* checks whether the probabilities are well calibrated (lower is better).

In [4]:
# Accuracy for each test season: is performance stable over time?
by_season = test.assign(logreg=logreg.predict(X_test), forest=forest.predict(X_test))
by_season.groupby('SEASON').apply(
    lambda g: pd.Series({'Logistic regression': (g.logreg == g.WIN).mean(),
                         'Random forest': (g.forest == g.WIN).mean()}),
    include_groups=False).round(3)

,Logistic regression,Random forest
SEASON,,
2023-24,0.959,0.936
2024-25,0.965,0.937
2025-26,0.956,0.918


## What this means

- **Both models crush the baseline**: 96% vs 55%. Even without points, these stats nearly decide the game.
- **Why so high?** The features come from *the same game* being predicted, so this model **explains** why a team won. It does not forecast games before they happen. The question it answers is *which stats matter*, not *who will win tomorrow*.
- **Logistic regression beats the random forest.** The links between these stats and winning are mostly straight lines (shoot better → win more), so the extra flexibility of the forest doesn't help, and the simpler model is easier to explain.
- **Results hold across all 3 test seasons** (92–97%), so the model isn't relying on one unusual year.